In [ ]:
!pip install librosa

In [ ]:
!pip install faiss-cpu tqdm librosa

In [ ]:
import librosa
import numpy as np

path = "/content/DJ Snake - Taki Taki ft_ Selena Gomez_ Ozuna_ Cardi B _Official Music Video_.wav"

# sr=None -> не ресемплить, бере оригінальний SR
audio_array, sr = librosa.load(path, sr=None, mono=True)   # audio_array shape: (N,)
audio_array = audio_array.astype(np.float32)

In [ ]:
import librosa
import numpy as np
import torch
from transformers import ClapModel, ClapProcessor

path = "/content/DJ Snake - Taki Taki ft_ Selena Gomez_ Ozuna_ Cardi B _Official Music Video_.wav"

audio_array, sr = librosa.load(path, sr=48000, mono=True)  # <-- 48k
audio_array = audio_array.astype(np.float32)

model = ClapModel.from_pretrained("laion/clap-htsat-unfused")
processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")

inputs = processor(audios=audio_array, sampling_rate=48000, return_tensors="pt")
with torch.no_grad():
    emb = model.get_audio_features(**inputs)  # (1, D)

print(audio_array.shape, sr, emb.shape)


/tmp/ipython-input-315372304.py:14: FutureWarning: `audios` is deprecated and will be removed in version v4.59.0 for `ClapProcessor.__call__`. Use `audio` instead.
  inputs = processor(audios=audio_array, sampling_rate=48000, return_tensors="pt")


(1432373,) 48000 torch.Size([1, 512])


In [ ]:
import numpy as np
import librosa
import torch
from transformers import ClapModel, ClapProcessor

MODEL_ID = "laion/larger_clap_music"  # :contentReference[oaicite:2]{index=2}

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ClapModel.from_pretrained(MODEL_ID).to(device)
processor = ClapProcessor.from_pretrained(MODEL_ID)

def clap_embed_file(
    path: str,
    chunk_sec: float = 10.0,
    hop_sec: float = 10.0,
    target_sr: int = 48000
) -> torch.Tensor:
    """
    Повертає 1 ембединг (1, D) для всього треку:
    - ріже на чанки по chunk_sec
    - дістає ембединги для кожного чанку
    - усереднює (mean pooling)
    """
    # 1) load + resample to 48k (CLAP trained sampling rate)
    audio, sr = librosa.load(path, sr=target_sr, mono=True)
    audio = audio.astype(np.float32)

    chunk_len = int(chunk_sec * sr)
    hop_len = int(hop_sec * sr)

    # якщо трек короткий — просто один чанк
    if len(audio) <= chunk_len:
        chunks = [audio]
    else:
        chunks = []
        for start in range(0, len(audio) - chunk_len + 1, hop_len):
            chunks.append(audio[start:start + chunk_len])
        if not chunks:
            chunks = [audio]

    embs = []
    for ch in chunks:
        inputs = processor(audios=ch, sampling_rate=sr, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            e = model.get_audio_features(**inputs)  # (1, D)
        # L2-нормалізація часто корисна для cosine similarity
        e = torch.nn.functional.normalize(e, p=2, dim=-1)
        embs.append(e)

    # mean pooling по чанках -> (1, D)
    emb = torch.mean(torch.cat(embs, dim=0), dim=0, keepdim=True)
    emb = torch.nn.functional.normalize(emb, p=2, dim=-1)
    return emb

# приклад:
emb = clap_embed_file("/content/DJ Snake - Taki Taki ft_ Selena Gomez_ Ozuna_ Cardi B _Official Music Video_.wav", chunk_sec=15, hop_sec=15)
print(emb.shape)



/tmp/ipython-input-307867102.py:43: FutureWarning: `audios` is deprecated and will be removed in version v4.59.0 for `ClapProcessor.__call__`. Use `audio` instead.
  inputs = processor(audios=ch, sampling_rate=sr, return_tensors="pt")


torch.Size([1, 512])


In [ ]:
!pip install faiss-cpu tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.6 MB/s eta 0:00:00


In [ ]:
emb

tensor([[-1.5402e-02, -4.6850e-03,  1.0557e-01, -4.8764e-02, -4.1966e-02,
          4.5696e-02, -4.5988e-02, -4.0983e-02, -3.8440e-02,  5.1099e-02,
          3.9821e-02, -9.0425e-03, -9.1835e-02,  7.8176e-03,  1.0970e-02,
         -2.7160e-02,  6.2741e-02,  1.0129e-01, -4.9742e-02, -2.6188e-02,
          4.3088e-02,  2.8040e-02,  4.2691e-02, -5.5365e-02, -5.5161e-02,
          3.6653e-02, -1.1490e-02, -7.4489e-03, -2.1373e-02,  5.0903e-02,
          1.7283e-02,  1.1841e-01, -8.7159e-02, -6.9057e-02, -1.9590e-03,
         -5.7204e-02, -5.3761e-02,  2.9631e-02, -3.3466e-02,  2.1437e-02,
          8.2102e-02,  1.9830e-02,  1.1400e-02, -1.4441e-02, -6.4951e-02,
         -2.6834e-02, -5.4093e-02,  4.0348e-02, -6.3209e-02, -2.1003e-02,
         -2.7166e-02,  1.9991e-02,  2.0374e-03,  9.4368e-03, -3.7950e-02,
          1.1234e-02, -3.4662e-03, -4.9589e-02,  1.9836e-02,  4.1801e-02,
          6.5884e-02, -3.4308e-02,  6.3441e-03, -5.6542e-02, -6.7351e-02,
         -3.2800e-02, -4.0851e-03,  5.

In [ ]:
!unzip -q "original.zip" -d "/content/original"
!ls -la /content/original | head


total 12
drwxr-xr-x 3 root root 4096 Jan 28 21:39 .
drwxr-xr-x 1 root root 4096 Jan 28 21:39 ..
drwxr-xr-x 2 root root 4096 Jan 28 08:11 original


In [ ]:
!unzip -q "comparison.zip" -d "/content/comparison"
!ls -la /content/comparison | head

total 12
drwxr-xr-x 3 root root 4096 Jan 28 21:39 .
drwxr-xr-x 1 root root 4096 Jan 28 21:39 ..
drwxr-xr-x 2 root root 4096 Jan 28 08:10 comparison


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ВАРІАНТ БЕЗ НОРМАЛІЗАЦІЇ:
# - у CSV колонки: ori_title, comp_title
# - значення у цих колонках мають містити розширення ".wav"
# - файли лежать у /content/origin та /content/comparison
# - будуємо FAISS-індекс по comparison і для кожного ori_title шукаємо top-1
#
# pip install -U transformers torch librosa faiss-cpu pandas tqdm soundfile

from pathlib import Path
import json
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn.functional as F
from tqdm import tqdm
import faiss
from transformers import ClapModel, ClapProcessor

# --------- Налаштуй ці шляхи ----------
ORIGIN_DIR = Path("/content/original/original")
COMPARISON_DIR = Path("/content/comparison/comparison")
PAIRS_CSV = Path("/content/song_pairs.csv")  # або "/content/song_pairs.csv"

MODEL_ID = "laion/larger_clap_music"
TARGET_SR = 48000
CHUNK_SEC = 15.0
HOP_SEC = 15.0

CACHE_DIR = Path("./cache_clap")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---- важливо: працюємо ТІЛЬКИ з .wav ----
AUDIO_EXT = ".wav"


def ensure_wav(name: str) -> str:
    """Гарантує, що назва закінчується .wav (інакше додає)."""
    name = str(name)
    return name if name.lower().endswith(AUDIO_EXT) else name + AUDIO_EXT


def list_wav_files(folder: Path):
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() == AUDIO_EXT])


def name_to_path_map(folder: Path):
    """
    Мапа точна: 'filename.wav' -> Path
    (без stem, без нормалізації)
    """
    mp = {}
    for p in list_wav_files(folder):
        mp[p.name] = p
    return mp


def load_audio_mono_48k(path: str):
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    audio = audio.astype(np.float32)
    audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)
    return audio, sr


@torch.no_grad()
def clap_embed_track(model, processor, audio: np.ndarray, sr: int, device: str) -> np.ndarray:
    """Ембединг (D,) для треку: chunk -> embed -> mean pooling -> L2 norm."""
    chunk_len = int(CHUNK_SEC * sr)
    hop_len = int(HOP_SEC * sr)

    if len(audio) <= chunk_len:
        chunks = [audio]
    else:
        chunks = [audio[i:i+chunk_len] for i in range(0, len(audio)-chunk_len+1, hop_len)]
        if not chunks:
            chunks = [audio]

    embs = []
    for ch in chunks:
        inputs = processor(audios=ch, sampling_rate=sr, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        e = model.get_audio_features(**inputs)  # (1, D)
        e = F.normalize(e, p=2, dim=-1)
        embs.append(e)

    emb = torch.mean(torch.cat(embs, dim=0), dim=0)  # (D,)
    emb = F.normalize(emb, p=2, dim=-1)
    return emb.cpu().numpy().astype("float32")


def embed_comparison_folder_with_cache(folder: Path, model, processor, device: str):
    """
    Ембедить всі .wav з папки comparison, кешує.
    Повертає:
      X: (N, D) float32 L2-норм
      meta: список dict {name, path}
    """
    x_path = CACHE_DIR / "comparison_X.npy"
    m_path = CACHE_DIR / "comparison_meta.json"

    if x_path.exists() and m_path.exists():
        X = np.load(x_path)
        meta = json.loads(m_path.read_text(encoding="utf-8"))
        return X, meta

    files = list_wav_files(folder)
    if not files:
        raise ValueError(f"Не знайдено .wav у папці: {folder}")

    embs, meta = [], []
    for p in tqdm(files, desc="Embedding comparison"):
        try:
            audio, sr = load_audio_mono_48k(str(p))
            emb = clap_embed_track(model, processor, audio, sr, device)
            embs.append(emb)
            meta.append({"name": p.name, "path": str(p)})
        except Exception as e:
            print(f"[SKIP] {p.name} -> {type(e).__name__}: {e}")

    if not embs:
        raise RuntimeError("Не створено жодного ембединга для comparison.")

    X = np.vstack(embs).astype("float32")
    X /= (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)

    np.save(x_path, X)
    m_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    return X, meta


def build_faiss_index_ip(X: np.ndarray):
    """Cosine similarity через inner product (бо X L2-норм)."""
    d = X.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(X)
    return index


# ---------------- MAIN ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = ClapModel.from_pretrained(MODEL_ID).to(device).eval()
processor = ClapProcessor.from_pretrained(MODEL_ID)

# 1) БД по comparison
X_comp, meta_comp = embed_comparison_folder_with_cache(COMPARISON_DIR, model, processor, device)
index = build_faiss_index_ip(X_comp)

# 2) Мапи точних імен .wav -> шлях
origin_map = name_to_path_map(ORIGIN_DIR)
comp_map = name_to_path_map(COMPARISON_DIR)

# 3) Пари з CSV
df = pd.read_csv(PAIRS_CSV)
if "ori_title" not in df.columns or "comp_title" not in df.columns:
    raise ValueError(f"CSV має містити колонки ori_title та comp_title. Є: {list(df.columns)}")

results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Checking pairs"):
    ori_title = ensure_wav(row["ori_title"])
    comp_title = ensure_wav(row["comp_title"])

    ori_path = origin_map.get(ori_title)
    print(ori_path)
    expected_path = comp_map.get(comp_title)

    if ori_path is None:
        results.append({
            "row": i,
            "ori_title": ori_title,
            "comp_title": comp_title,
            "status": "ORI_NOT_FOUND",
            "pred_name": None,
            "pred_path": None,
            "score_cosine": None,
            "expected_path": str(expected_path) if expected_path else None,
            "match": False,
        })
        continue

    # query embed
    audio, sr = load_audio_mono_48k(str(ori_path))
    q = clap_embed_track(model, processor, audio, sr, device).reshape(1, -1)
    q /= (np.linalg.norm(q, axis=1, keepdims=True) + 1e-12)

    scores, ids = index.search(q, 1)
    best_score = float(scores[0][0])
    best_idx = int(ids[0][0])

    pred_name = meta_comp[best_idx]["name"]   # це "xxx.wav"
    pred_path = meta_comp[best_idx]["path"]

    # перевірка відповідності: точна рівність імен .wav
    ok = (pred_name == comp_title)

    results.append({
        "row": i,
        "ori_title": ori_title,
        "comp_title": comp_title,
        "status": "OK" if expected_path is not None else "EXPECTED_NOT_FOUND",
        "ori_path": str(ori_path),
        "pred_name": pred_name,
        "pred_path": pred_path,
        "score_cosine": best_score,
        "expected_path": str(expected_path) if expected_path else None,
        "match": bool(ok),
    })

res_df = pd.DataFrame(results)

valid = res_df[res_df["status"].isin(["OK", "EXPECTED_NOT_FOUND"])]
acc = float(valid["match"].mean()) if len(valid) else float("nan")

print("\n--- SUMMARY ---")
print("Total rows:", len(res_df))
print("Evaluated:", len(valid))
print("Accuracy:", acc)

print("\n--- MISMATCHES (top 20 by similarity) ---")
bad = valid[valid["match"] == False].sort_values("score_cosine", ascending=False)
display(bad.head(20))

out_path = Path("./pair_check_results_wav_exact.csv")
res_df.to_csv(out_path, index=False, encoding="utf-8")
print("\nSaved:", out_path.resolve())


Device: cpu


Checking pairs:   0%|          | 0/16 [00:00<?, ?it/s]/tmp/ipython-input-1630153378.py:80: FutureWarning: `audios` is deprecated and will be removed in version v4.59.0 for `ClapProcessor.__call__`. Use `audio` instead.
  inputs = processor(audios=ch, sampling_rate=sr, return_tensors="pt")


/content/original/The Gap Band - Oops Upside Your Head.wav


Checking pairs:   6%|▋         | 1/16 [00:01<00:18,  1.24s/it]

/content/original/Jennifer Rush - The Power Of Love _Official Video_ _VOD_.wav


...

/content/original/The Monkees _ Mary_ Mary.wav


Checking pairs: 100%|██████████| 16/16 [00:30<00:00,  1.92s/it]


--- SUMMARY ---
Total rows: 16
Evaluated: 15
Accuracy: 0.13333333333333333

--- MISMATCHES (top 20 by similarity) ---


,row,ori_title,comp_title,status,ori_path,pred_name,pred_path,score_cosine,expected_path,match
14,14,Right Said Fred - I_m Too Sexy _Original Mix -...,Taylor Swift - Look What You Made Me Do.wav,OK,/content/original/Right Said Fred - I_m Too Se...,Vanilla Ice - Ice Ice Baby _Official Music Vid...,/content/comparison/comparison/Vanilla Ice - I...,0.996852,/content/comparison/comparison/Taylor Swift - ...,False
12,12,Gloria Gaynor-I Will Survive.wav,진주-난 괜찮아.wav,EXPECTED_NOT_FOUND,/content/original/Gloria Gaynor-I Will Survive...,Vanilla Ice - Ice Ice Baby _Official Music Vid...,/content/comparison/comparison/Vanilla Ice - I...,0.995887,None,False
1,1,Jennifer Rush - The Power Of Love _Official Vi...,Céline Dion - The Power Of Love _Official Rema...,EXPECTED_NOT_FOUND,/content/original/Jennifer Rush - The Power Of...,Céline Dion - The Power Of Love _Official Rem...,/content/comparison/comparison/Céline Dion - ...,0.994556,None,False
13,13,Carmen - Habanera.wav,I-DLE - Nxde.wav,OK,/content/original/Carmen - Habanera.wav,Céline Dion - The Power Of Love _Official Rem...,/content/comparison/comparison/Céline Dion - ...,0.994299,/content/comparison/comparison/I-DLE - Nxde.wav,False
6,6,Astrud Gilberto - Maria Quiet.wav,Smoke On The Water _2024 Remastered_.wav,OK,/content/original/Astrud Gilberto - Maria Quie...,ТНМК _ _Люба_ Люба_.wav,/content/comparison/comparison/ТНМК _ _Люба_ Л...,0.993453,/content/comparison/comparison/Smoke On The Wa...,False
9,9,September - Cry For You.wav,Ed Sheeran - Bad Habits _Official Lyric Video_...,OK,/content/original/September - Cry For You.wav,Juice WRLD - Lucid Dreams _Official Music Vide...,/content/comparison/comparison/Juice WRLD - Lu...,0.993389,/content/comparison/comparison/Ed Sheeran - Ba...,False
5,5,Joyful Noise - Official Music Video_ FLAME fea...,Katy Perry - Dark Horse _Lyrics_ ft_ Juicy J.wav,OK,/content/original/Joyful Noise - Official Musi...,Vanilla Ice - Ice Ice Baby _Official Music Vid...,/content/comparison/comparison/Vanilla Ice - I...,0.993215,/content/comparison/comparison/Katy Perry - Da...,False
4,4,Queen - Under Pressure _Official Video_.wav,Vanilla Ice - Ice Ice Baby _Official Music Vid...,OK,/content/original/Queen - Under Pressure _Offi...,Mark Ronson - Uptown Funk _Official Video_ ft_...,/content/comparison/comparison/Mark Ronson - U...,0.992826,/content/comparison/comparison/Vanilla Ice - I...,False
11,11,The Police - Message In A Bottle _Official Mus...,Rihanna-Love without Tragedy_only_.wav,OK,/content/original/The Police - Message In A Bo...,Juice WRLD - Lucid Dreams _Official Music Vide...,/content/comparison/comparison/Juice WRLD - Lu...,0.990140,/content/comparison/comparison/Rihanna-Love wi...,False
15,15,The Monkees _ Mary_ Mary.wav,ТНМК _ _Люба_ Люба_.wav,OK,/content/original/The Monkees _ Mary_ Mary.wav,Mariah Carey - Without You _Official Lyric Vid...,/content/comparison/comparison/Mariah Carey - ...,0.989774,/content/comparison/comparison/ТНМК _ _Люба_ Л...,False



Saved: /content/pair_check_results_wav_exact.csv


In [ ]:
"""
Експеримент з пошуку схожих аудіофайлів через CLAP ембединги.

Скрипт порівнює оригінальні треки з базою comparison і визначає,
чи правильно знаходиться відповідний файл.

Запуск:
    pip install transformers torch librosa faiss-cpu pandas tqdm soundfile
    python audio_similarity_experiment.py
"""

import json
from pathlib import Path
from dataclasses import dataclass
from typing import Optional
import itertools

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn.functional as F
from tqdm import tqdm
import faiss
from transformers import ClapModel, ClapProcessor


# ============================================================
# НАЛАШТУВАННЯ - змінюй тут!
# ============================================================

@dataclass
class Config:

    origin_dir: Path = Path("/content/original/original")
    comparison_dir: Path = Path("/content/comparison/comparison")
    pairs_csv: Path = Path("/content/song_pairs.csv")
    output_dir: Path = Path("./results")

    # Модель CLAP
    model_id: str = "laion/larger_clap_music"

    # Параметри обробки аудіо
    sample_rate: int = 48000
    chunk_seconds: float = 15.0  # довжина чанку
    hop_seconds: float = 15.0    # крок між чанками

    # Кешування
    use_cache: bool = True
    cache_dir: Path = Path("./cache_clap")


# ============================================================
# ПАРАМЕТРИ ДЛЯ ЕКСПЕРИМЕНТІВ
# ============================================================

EXPERIMENT_PARAMS = {
    # Різні розміри чанків (в секундах)
    "chunk_seconds": [5.0, 10.0, 15.0, 20.0, 30.0],

    # Різні кроки (hop) - може бути overlap якщо hop < chunk
    "hop_seconds": [5.0, 10.0, 15.0],

    # Різні sample rates
    "sample_rate": [16000, 22050, 48000],
}


# ============================================================
# ДОПОМІЖНІ ФУНКЦІЇ
# ============================================================

def ensure_wav_extension(name: str) -> str:
    """Додає .wav якщо нема."""
    name = str(name)
    return name if name.lower().endswith(".wav") else name + ".wav"


def find_wav_files(folder: Path) -> list[Path]:
    """Знаходить всі .wav файли в папці (рекурсивно)."""
    return sorted(p for p in folder.rglob("*.wav") if p.is_file())


def build_filename_map(folder: Path) -> dict[str, Path]:
    """Створює мапу: ім'я файлу -> повний шлях."""
    return {p.name: p for p in find_wav_files(folder)}


# ============================================================
# РОБОТА З АУДІО
# ============================================================

def load_audio(path: Path, sample_rate: int) -> np.ndarray:
    """Завантажує аудіо, конвертує в моно."""
    audio, _ = librosa.load(str(path), sr=sample_rate, mono=True)
    audio = audio.astype(np.float32)
    audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)
    return audio


def split_into_chunks(audio: np.ndarray, sample_rate: int,
                      chunk_sec: float, hop_sec: float) -> list[np.ndarray]:
    """Розбиває аудіо на чанки з заданим кроком."""
    chunk_len = int(chunk_sec * sample_rate)
    hop_len = int(hop_sec * sample_rate)

    if len(audio) <= chunk_len:
        return [audio]

    chunks = []
    for start in range(0, len(audio) - chunk_len + 1, hop_len):
        chunks.append(audio[start:start + chunk_len])

    return chunks if chunks else [audio]


# ============================================================
# CLAP ЕМБЕДИНГИ
# ============================================================

class CLAPEmbedder:
    """Обгортка для CLAP моделі."""

    def __init__(self, model_id: str, device: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Використовую пристрій: {self.device}")

        print(f"Завантажую модель: {model_id}")
        self.model = ClapModel.from_pretrained(model_id).to(self.device).eval()
        self.processor = ClapProcessor.from_pretrained(model_id)

    @torch.no_grad()
    def embed_audio(self, audio: np.ndarray, sample_rate: int,
                    chunk_sec: float, hop_sec: float) -> np.ndarray:
        """
        Створює ембединг для аудіо.

        Процес:
        1. Розбиваємо на чанки
        2. Для кожного чанка отримуємо ембединг
        3. Усереднюємо всі ембединги
        4. Нормалізуємо результат
        """
        chunks = split_into_chunks(audio, sample_rate, chunk_sec, hop_sec)

        embeddings = []
        for chunk in chunks:
            inputs = self.processor(audios=chunk, sampling_rate=sample_rate,
                                   return_tensors="pt")
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            emb = self.model.get_audio_features(**inputs)
            emb = F.normalize(emb, p=2, dim=-1)
            embeddings.append(emb)

        # Усереднення всіх чанків
        mean_emb = torch.mean(torch.cat(embeddings, dim=0), dim=0)
        mean_emb = F.normalize(mean_emb, p=2, dim=-1)

        return mean_emb.cpu().numpy().astype("float32")


# ============================================================
# FAISS ІНДЕКС ДЛЯ ПОШУКУ
# ============================================================

class AudioSearchIndex:
    """Індекс для швидкого пошуку схожих аудіо."""

    def __init__(self):
        self.index: Optional[faiss.IndexFlatIP] = None
        self.metadata: list[dict] = []

    def build(self, embeddings: np.ndarray, metadata: list[dict]):
        """Будує індекс з ембедингів."""
        # Нормалізуємо для cosine similarity
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        embeddings = embeddings / (norms + 1e-12)

        # IndexFlatIP = inner product = cosine sim для нормалізованих векторів
        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)
        self.metadata = metadata

        print(f"Індекс побудовано: {len(metadata)} елементів")

    def search(self, query: np.ndarray, top_k: int = 1) -> list[tuple[dict, float]]:
        """Шукає найближчі елементи."""
        query = query.reshape(1, -1)
        query = query / (np.linalg.norm(query) + 1e-12)

        scores, indices = self.index.search(query, top_k)

        results = []
        for idx, score in zip(indices[0], scores[0]):
            results.append((self.metadata[idx], float(score)))

        return results


# ============================================================
# КЕШУВАННЯ
# ============================================================

class EmbeddingCache:
    """Кеш для ембедингів."""

    def __init__(self, cache_dir: Path, prefix: str):
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.emb_path = cache_dir / f"{prefix}_embeddings.npy"
        self.meta_path = cache_dir / f"{prefix}_metadata.json"

    def exists(self) -> bool:
        return self.emb_path.exists() and self.meta_path.exists()

    def save(self, embeddings: np.ndarray, metadata: list[dict]):
        np.save(self.emb_path, embeddings)
        self.meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2))

    def load(self) -> tuple[np.ndarray, list[dict]]:
        embeddings = np.load(self.emb_path)
        metadata = json.loads(self.meta_path.read_text())
        return embeddings, metadata


# ============================================================
# ГОЛОВНИЙ ЕКСПЕРИМЕНТ
# ============================================================

def embed_folder(folder: Path, embedder: CLAPEmbedder, config: Config,
                 cache: Optional[EmbeddingCache] = None) -> tuple[np.ndarray, list[dict]]:
    """Ембедить всі файли в папці."""

    # Пробуємо завантажити з кешу
    if cache and cache.exists() and config.use_cache:
        print(f"Завантажую з кешу: {cache.emb_path}")
        return cache.load()

    files = find_wav_files(folder)
    if not files:
        raise ValueError(f"Не знайдено .wav файлів у {folder}")

    print(f"Знайдено {len(files)} файлів")

    embeddings = []
    metadata = []

    for path in tqdm(files, desc="Ембединг"):
        try:
            audio = load_audio(path, config.sample_rate)
            emb = embedder.embed_audio(audio, config.sample_rate,
                                       config.chunk_seconds, config.hop_seconds)
            embeddings.append(emb)
            metadata.append({"name": path.name, "path": str(path)})
        except Exception as e:
            print(f"[ПРОПУСК] {path.name}: {e}")

    if not embeddings:
        raise RuntimeError("Не створено жодного ембедингу")

    embeddings = np.vstack(embeddings).astype("float32")

    # Зберігаємо в кеш
    if cache:
        cache.save(embeddings, metadata)
        print(f"Збережено в кеш: {cache.emb_path}")

    return embeddings, metadata


def run_experiment(config: Config) -> pd.DataFrame:
    """Запускає один експеримент з заданими параметрами."""

    print("\n" + "=" * 60)
    print(f"ЕКСПЕРИМЕНТ")
    print(f"  chunk_sec={config.chunk_seconds}, hop_sec={config.hop_seconds}")
    print(f"  sample_rate={config.sample_rate}")
    print("=" * 60)

    # Ініціалізація
    embedder = CLAPEmbedder(config.model_id)

    # Кеш для comparison (з унікальним ключем параметрів)
    cache_key = f"comp_sr{config.sample_rate}_ch{config.chunk_seconds}_hp{config.hop_seconds}"
    cache = EmbeddingCache(config.cache_dir, cache_key) if config.use_cache else None

    # Ембединг comparison папки
    comp_embs, comp_meta = embed_folder(config.comparison_dir, embedder, config, cache)

    # Будуємо індекс
    search_index = AudioSearchIndex()
    search_index.build(comp_embs, comp_meta)

    # Мапи файлів
    origin_map = build_filename_map(config.origin_dir)
    comp_map = build_filename_map(config.comparison_dir)

    # Читаємо пари
    df = pd.read_csv(config.pairs_csv)
    if "ori_title" not in df.columns or "comp_title" not in df.columns:
        raise ValueError("CSV має містити колонки: ori_title, comp_title")

    # Перевірка пар
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Перевірка пар"):
        ori_name = ensure_wav_extension(row["ori_title"])
        comp_name = ensure_wav_extension(row["comp_title"])

        ori_path = origin_map.get(ori_name)
        expected_path = comp_map.get(comp_name)

        # Файл не знайдено
        if ori_path is None:
            results.append({
                "ori_title": ori_name,
                "comp_title": comp_name,
                "status": "ORI_NOT_FOUND",
                "predicted": None,
                "score": None,
                "match": False,
            })
            continue

        # Ембедимо та шукаємо
        audio = load_audio(ori_path, config.sample_rate)
        query = embedder.embed_audio(audio, config.sample_rate,
                                     config.chunk_seconds, config.hop_seconds)

        matches = search_index.search(query, top_k=1)
        best_match, score = matches[0]

        is_correct = (best_match["name"] == comp_name)

        results.append({
            "ori_title": ori_name,
            "comp_title": comp_name,
            "status": "OK",
            "predicted": best_match["name"],
            "score": score,
            "match": is_correct,
        })

    return pd.DataFrame(results)


def print_summary(results: pd.DataFrame, config: Config):
    """Виводить підсумок експерименту."""
    valid = results[results["status"] == "OK"]

    if len(valid) == 0:
        print("Немає валідних результатів!")
        return

    accuracy = valid["match"].mean()
    avg_score = valid["score"].mean()

    print(f"\n--- РЕЗУЛЬТАТИ ---")
    print(f"Всього пар: {len(results)}")
    print(f"Перевірено: {len(valid)}")
    print(f"Точність: {accuracy:.2%}")
    print(f"Середній score: {avg_score:.4f}")

    # Помилки
    errors = valid[~valid["match"]].sort_values("score", ascending=False)
    if len(errors) > 0:
        print(f"\nПомилки ({len(errors)}):")
        print(errors[["ori_title", "comp_title", "predicted", "score"]].head(10).to_string())


def run_grid_search(base_config: Config, params: dict) -> pd.DataFrame:
    """
    Запускає серію експериментів з різними параметрами.

    Приклад params:
    {
        "chunk_seconds": [5.0, 10.0, 15.0],
        "hop_seconds": [5.0, 10.0],
    }
    """
    # Створюємо всі комбінації
    keys = list(params.keys())
    values = list(params.values())
    combinations = list(itertools.product(*values))

    print(f"Запускаю {len(combinations)} експериментів...")

    all_results = []

    for combo in combinations:
        # Створюємо конфіг для цієї комбінації
        config = Config(
            origin_dir=base_config.origin_dir,
            comparison_dir=base_config.comparison_dir,
            pairs_csv=base_config.pairs_csv,
            output_dir=base_config.output_dir,
            model_id=base_config.model_id,
            sample_rate=base_config.sample_rate,
            chunk_seconds=base_config.chunk_seconds,
            hop_seconds=base_config.hop_seconds,
            use_cache=base_config.use_cache,
            cache_dir=base_config.cache_dir,
        )

        # Застосовуємо параметри
        for key, value in zip(keys, combo):
            setattr(config, key, value)

        # Запускаємо
        try:
            results = run_experiment(config)
            valid = results[results["status"] == "OK"]

            accuracy = valid["match"].mean() if len(valid) > 0 else 0
            avg_score = valid["score"].mean() if len(valid) > 0 else 0
            print( "accuracy " + str(accuracy))

            all_results.append({
                **{k: v for k, v in zip(keys, combo)},
                "accuracy": accuracy,
                "avg_score": avg_score,
                "n_valid": len(valid),
            })

        except Exception as e:
            print(f"Помилка: {e}")
            all_results.append({
                **{k: v for k, v in zip(keys, combo)},
                "accuracy": None,
                "avg_score": None,
                "n_valid": 0,
                "error": str(e),
            })

    return pd.DataFrame(all_results)


# ============================================================
# ТОЧКА ВХОДУ
# ============================================================

# config = Config()
# config.output_dir.mkdir(parents=True, exist_ok=True)

# results = run_experiment(config)
# print_summary(results, config)

# output_file = config.output_dir / "single_experiment.csv"
# results.to_csv(output_file, index=False)
# print(f"\nЗбережено: {output_file}")

# grid_results = run_grid_search(config, {
#         "chunk_seconds": [5.0, 10.0, 15.0, 20.0],
#         "hop_seconds": [5.0, 10.0, 15.0],
#     })

# print(grid_results.sort_values("accuracy", ascending=False).to_string())

# grid_file = config.output_dir / "grid_search_results.csv"
# grid_results.to_csv(grid_file, index=False)
# print(f"\nЗбережено: {grid_file}")


In [ ]:
config = Config(hop_seconds = 15,chunk_seconds =5  )
config.output_dir.mkdir(parents=True, exist_ok=True)

results = run_experiment(config)
print_summary(results, config)



ЕКСПЕРИМЕНТ
  chunk_sec=5, hop_sec=15
  sample_rate=48000
Використовую пристрій: cpu
Завантажую модель: laion/larger_clap_music
Завантажую з кешу: cache_clap/comp_sr48000_ch5_hp15_embeddings.npy
Індекс побудовано: 16 елементів


Перевірка пар:   0%|          | 0/16 [00:00<?, ?it/s]/tmp/ipython-input-364490381.py:148: FutureWarning: `audios` is deprecated and will be removed in version v4.59.0 for `ClapProcessor.__call__`. Use `audio` instead.
  inputs = self.processor(audios=chunk, sampling_rate=sample_rate,
Перевірка пар: 100%|██████████| 16/16 [00:43<00:00,  2.70s/it]


--- РЕЗУЛЬТАТИ ---
Всього пар: 16
Перевірено: 15
Точність: 26.67%
Середній score: 0.9892

Помилки (11):
                                                                           ori_title                                                                        comp_title                                                            predicted     score
14                  Right Said Fred - I_m Too Sexy _Original Mix - 2006 Version_.wav                                       Taylor Swift - Look What You Made Me Do.wav                Vanilla Ice - Ice Ice Baby _Official Music Video_.wav  0.995461
13                                                             Carmen - Habanera.wav                                                                  I-DLE - Nxde.wav  Céline Dion - The Power Of Love _Official Remastered HD Video_.wav  0.994866
6                                                  Astrud Gilberto - Maria Quiet.wav                                          Smoke On The Water _2024 Remast

In [ ]:
res_df[res_df['match'] == True]

,row,ori_title,comp_title,status,ori_path,pred_name,pred_path,score_cosine,expected_path,match
2,2,Without you - Badfinger.wav,Mariah Carey - Without You _Official Lyric Vid...,OK,/content/original/Without you - Badfinger.wav,Mariah Carey - Without You _Official Lyric Vid...,/content/comparison/comparison/Mariah Carey - ...,0.988991,/content/comparison/comparison/Mariah Carey - ...,True
8,8,Sting - Shape of My Heart _Official Music Vide...,Juice WRLD - Lucid Dreams _Official Music Vide...,OK,/content/original/Sting - Shape of My Heart _O...,Juice WRLD - Lucid Dreams _Official Music Vide...,/content/comparison/comparison/Juice WRLD - Lu...,0.997416,/content/comparison/comparison/Juice WRLD - Lu...,True


In [ ]:
"""
Гібридний пошук схожих аудіо: FAISS (швидко) + DTW (точно).

Як працює:
1. CLAP перетворює аудіо в послідовність ембедингів (по чанках)
2. FAISS швидко знаходить кандидатів по усередненому вектору
3. DTW точно порівнює послідовності серед кандидатів

Запуск:
    pip install transformers torch librosa faiss-cpu pandas tqdm soundfile scipy
    python hybrid_audio_search.py
"""

import json
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn.functional as F
from tqdm import tqdm
import faiss
from scipy.spatial.distance import cdist
from transformers import ClapModel, ClapProcessor


# ============================================================
# НАЛАШТУВАННЯ
# ============================================================

@dataclass
class Config:
    # Шляхи
    origin_dir: Path = Path("/content/original/original")
    comparison_dir: Path = Path("/content/comparison/comparison")
    pairs_csv: Path = Path("/content/song_pairs.csv")
    cache_dir: Path = Path("./cache")

    # CLAP модель
    model_id: str = "laion/larger_clap_music"

    # Аудіо
    sample_rate: int = 48000
    chunk_sec: float = 10.0
    hop_sec: float = 5.0  # overlap для кращого покриття

    # Гібридний пошук
    faiss_candidates: int = 50  # скільки кандидатів бере FAISS

    # Кеш
    use_cache: bool = True


# ============================================================
# РОБОТА З АУДІО
# ============================================================

def load_audio(path: Path, sr: int) -> np.ndarray:
    """Завантажує аудіо як моно float32."""
    audio, _ = librosa.load(str(path), sr=sr, mono=True)
    return np.nan_to_num(audio.astype(np.float32))


def split_chunks(audio: np.ndarray, sr: int, chunk_sec: float, hop_sec: float) -> list:
    """Розбиває аудіо на чанки з перекриттям."""
    chunk_len = int(chunk_sec * sr)
    hop_len = int(hop_sec * sr)

    if len(audio) <= chunk_len:
        return [audio]

    chunks = []
    for start in range(0, len(audio) - chunk_len + 1, hop_len):
        chunks.append(audio[start:start + chunk_len])

    return chunks if chunks else [audio]


# ============================================================
# CLAP ЕМБЕДИНГИ
# ============================================================

class CLAPEncoder:
    """Кодує аудіо в ембединги через CLAP."""

    def __init__(self, model_id: str):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Device: {self.device}")

        self.model = ClapModel.from_pretrained(model_id).to(self.device).eval()
        self.processor = ClapProcessor.from_pretrained(model_id)

    @torch.no_grad()
    def encode_chunk(self, chunk: np.ndarray, sr: int) -> np.ndarray:
        """Один чанк -> один ембединг (D,)."""
        inputs = self.processor(audios=chunk, sampling_rate=sr, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        emb = self.model.get_audio_features(**inputs)
        emb = F.normalize(emb, p=2, dim=-1)

        return emb.cpu().numpy().squeeze().astype("float32")

    def encode_track(self, audio: np.ndarray, sr: int,
                     chunk_sec: float, hop_sec: float) -> tuple:
        """
        Трек -> (sequence, mean).

        sequence: (n_chunks, D) — послідовність ембедингів
        mean: (D,) — усереднений ембединг для FAISS
        """
        chunks = split_chunks(audio, sr, chunk_sec, hop_sec)

        embeddings = [self.encode_chunk(ch, sr) for ch in chunks]
        sequence = np.stack(embeddings)

        mean = sequence.mean(axis=0)
        mean = mean / (np.linalg.norm(mean) + 1e-12)

        return sequence, mean


# ============================================================
# DTW (Dynamic Time Warping)
# ============================================================

def dtw_distance(seq1: np.ndarray, seq2: np.ndarray) -> float:
    """
    Відстань DTW між двома послідовностями.

    seq1: (n, D)
    seq2: (m, D)

    DTW знаходить оптимальне вирівнювання послідовностей,
    навіть якщо вони різної довжини або зсунуті.
    """
    n, m = len(seq1), len(seq2)

    # Cosine відстань між кожною парою чанків
    cost = cdist(seq1, seq2, metric="cosine")

    # DP: шукаємо шлях з мінімальною сумою
    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i, j] = cost[i-1, j-1] + min(
                dp[i-1, j],      # пропуск в seq2
                dp[i, j-1],      # пропуск в seq1
                dp[i-1, j-1]     # збіг
            )

    # Нормалізуємо по довжині шляху
    return dp[n, m] / (n + m)


# ============================================================
# ГІБРИДНИЙ ПОШУК
# ============================================================

class HybridIndex:
    """
    Гібридний індекс: FAISS для швидкого відбору + DTW для точного ранжування.
    """

    def __init__(self, n_candidates: int = 50):
        self.n_candidates = n_candidates
        self.faiss_index = None
        self.sequences = []  # список (n_chunks, D) масивів
        self.metadata = []   # {"name": ..., "path": ...}

    def build(self, means: np.ndarray, sequences: list, metadata: list):
        """
        Будує індекс.

        means: (N, D) — усереднені ембединги для FAISS
        sequences: список з N масивів (n_chunks, D)
        metadata: список з N словників
        """
        # L2-нормалізація для cosine similarity через inner product
        norms = np.linalg.norm(means, axis=1, keepdims=True)
        means = means / (norms + 1e-12)

        self.faiss_index = faiss.IndexFlatIP(means.shape[1])
        self.faiss_index.add(means.astype("float32"))

        self.sequences = sequences
        self.metadata = metadata

        print(f"Індекс: {len(metadata)} треків")

    def search(self, query_mean: np.ndarray, query_seq: np.ndarray) -> tuple:
        """
        Пошук найближчого треку.

        1. FAISS вибирає top-N кандидатів по mean ембедингу
        2. DTW ранжує кандидатів по послідовності

        Повертає: (metadata, similarity_score)
        """
        # Нормалізуємо запит
        query_mean = query_mean / (np.linalg.norm(query_mean) + 1e-12)
        query_mean = query_mean.reshape(1, -1).astype("float32")

        # Крок 1: FAISS — швидкий відбір кандидатів
        n_search = min(self.n_candidates, len(self.sequences))
        _, indices = self.faiss_index.search(query_mean, n_search)
        candidate_ids = indices[0]

        # Крок 2: DTW — точне порівняння послідовностей
        best_idx = None
        best_dist = float("inf")

        for idx in candidate_ids:
            dist = dtw_distance(query_seq, self.sequences[idx])
            if dist < best_dist:
                best_dist = dist
                best_idx = idx

        # Конвертуємо відстань в similarity (0-1, більше = краще)
        similarity = 1.0 / (1.0 + best_dist)

        return self.metadata[best_idx], similarity


# ============================================================
# КЕШУВАННЯ
# ============================================================

def save_cache(cache_dir: Path, means: np.ndarray, sequences: list, metadata: list):
    """Зберігає ембединги в кеш."""
    cache_dir.mkdir(parents=True, exist_ok=True)

    np.save(cache_dir / "means.npy", means)
    np.save(cache_dir / "sequences.npy", np.array(sequences, dtype=object), allow_pickle=True)
    (cache_dir / "metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2)
    )


def load_cache(cache_dir: Path):
    """Завантажує ембединги з кешу."""
    means = np.load(cache_dir / "means.npy")
    sequences = list(np.load(cache_dir / "sequences.npy", allow_pickle=True))
    metadata = json.loads((cache_dir / "metadata.json").read_text())
    return means, sequences, metadata


def cache_exists(cache_dir: Path) -> bool:
    return all((cache_dir / f).exists() for f in ["means.npy", "sequences.npy", "metadata.json"])


# ============================================================
# ГОЛОВНА ЛОГІКА
# ============================================================

def find_wav_files(folder: Path) -> list:
    """Знаходить всі .wav файли."""
    return sorted(p for p in folder.rglob("*.wav") if p.is_file())


def build_name_map(folder: Path) -> dict:
    """Мапа: ім'я файлу -> шлях."""
    return {p.name: p for p in find_wav_files(folder)}


def ensure_wav(name: str) -> str:
    """Додає .wav якщо треба."""
    return name if name.lower().endswith(".wav") else name + ".wav"


def embed_folder(folder: Path, encoder: CLAPEncoder, cfg: Config) -> tuple:
    """
    Ембедить всі треки в папці.

    Повертає: (means, sequences, metadata)
    """
    # Пробуємо кеш
    if cfg.use_cache and cache_exists(cfg.cache_dir):
        print("Завантажую з кешу...")
        return load_cache(cfg.cache_dir)

    files = find_wav_files(folder)
    print(f"Знайдено {len(files)} файлів")

    means, sequences, metadata = [], [], []

    for path in tqdm(files, desc="Ембединг"):
        try:
            audio = load_audio(path, cfg.sample_rate)
            seq, mean = encoder.encode_track(audio, cfg.sample_rate,
                                              cfg.chunk_sec, cfg.hop_sec)
            sequences.append(seq)
            means.append(mean)
            metadata.append({"name": path.name, "path": str(path)})
        except Exception as e:
            print(f"[!] {path.name}: {e}")

    means = np.stack(means)

    # Зберігаємо в кеш
    if cfg.use_cache:
        save_cache(cfg.cache_dir, means, sequences, metadata)
        print(f"Збережено в кеш: {cfg.cache_dir}")

    return means, sequences, metadata


def run(cfg: Config):
    """Запускає експеримент."""

    print("=" * 50)
    print("ГІБРИДНИЙ ПОШУК: FAISS + DTW")
    print(f"chunk={cfg.chunk_sec}s, hop={cfg.hop_sec}s, candidates={cfg.faiss_candidates}")
    print("=" * 50)

    # Ініціалізація
    encoder = CLAPEncoder(cfg.model_id)

    # Ембединг comparison
    means, sequences, metadata = embed_folder(cfg.comparison_dir, encoder, cfg)

    # Будуємо індекс
    index = HybridIndex(n_candidates=cfg.faiss_candidates)
    index.build(means, sequences, metadata)

    # Мапи файлів
    origin_map = build_name_map(cfg.origin_dir)
    comp_map = build_name_map(cfg.comparison_dir)

    # Читаємо пари
    df = pd.read_csv(cfg.pairs_csv)

    # Перевіряємо
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Пошук"):
        ori_name = ensure_wav(row["ori_title"])
        comp_name = ensure_wav(row["comp_title"])

        ori_path = origin_map.get(ori_name)

        if ori_path is None:
            results.append({
                "ori": ori_name,
                "expected": comp_name,
                "predicted": None,
                "score": None,
                "match": False,
                "status": "NOT_FOUND"
            })
            continue

        # Ембедимо запит
        audio = load_audio(ori_path, cfg.sample_rate)
        query_seq, query_mean = encoder.encode_track(
            audio, cfg.sample_rate, cfg.chunk_sec, cfg.hop_sec
        )

        # Шукаємо
        match, score = index.search(query_mean, query_seq)

        results.append({
            "ori": ori_name,
            "expected": comp_name,
            "predicted": match["name"],
            "score": round(score, 4),
            "match": match["name"] == comp_name,
            "status": "OK"
        })

    # Результати
    results_df = pd.DataFrame(results)

    valid = results_df[results_df["status"] == "OK"]
    accuracy = valid["match"].mean() if len(valid) > 0 else 0

    print("\n" + "=" * 50)
    print("РЕЗУЛЬТАТИ")
    print("=" * 50)
    print(f"Всього: {len(results_df)}")
    print(f"Перевірено: {len(valid)}")
    print(f"Точність: {accuracy:.2%}")

    # Помилки
    errors = valid[~valid["match"]].sort_values("score", ascending=False)
    if len(errors) > 0:
        print(f"\nПомилки ({len(errors)}):")
        print(errors.head(10).to_string(index=False))

    # Зберігаємо
    output = Path("./hybrid_results.csv")
    results_df.to_csv(output, index=False)
    print(f"\nЗбережено: {output}")

    return results_df


# ============================================================
# ЗАПУСК
# ============================================================

cfg = Config()

    # Можна змінити параметри тут:
    # cfg.chunk_sec = 15.0
    # cfg.hop_sec = 7.5
    # cfg.faiss_candidates = 100

run(cfg)


ГІБРИДНИЙ ПОШУК: FAISS + DTW
chunk=10.0s, hop=5.0s, candidates=50
Device: cpu
Знайдено 16 файлів


Ембединг:   0%|          | 0/16 [00:00<?, ?it/s]/tmp/ipython-input-3191650063.py:98: FutureWarning: `audios` is deprecated and will be removed in version v4.59.0 for `ClapProcessor.__call__`. Use `audio` instead.
  inputs = self.processor(audios=chunk, sampling_rate=sr, return_tensors="pt")
Ембединг: 100%|██████████| 16/16 [01:44<00:00,  6.54s/it]


Збережено в кеш: cache
Індекс: 16 треків


Пошук:   0%|          | 0/16 [00:00<?, ?it/s]/tmp/ipython-input-3191650063.py:98: FutureWarning: `audios` is deprecated and will be removed in version v4.59.0 for `ClapProcessor.__call__`. Use `audio` instead.
  inputs = self.processor(audios=chunk, sampling_rate=sr, return_tensors="pt")
Пошук: 100%|██████████| 16/16 [01:28<00:00,  5.56s/it]


РЕЗУЛЬТАТИ
Всього: 16
Перевірено: 15
Точність: 13.33%

Помилки (13):
                                                                      ori                                                           expected                                                           predicted  score  match status
             Jennifer Rush - The Power Of Love _Official Video_ _VOD_.wav Céline Dion - The Power Of Love _Official Remastered HD Video_.wav Céline Dion - The Power Of Love _Official Remastered HD Video_.wav 0.9968  False     OK
                                                    Carmen - Habanera.wav                                                   I-DLE - Nxde.wav Céline Dion - The Power Of Love _Official Remastered HD Video_.wav 0.9960  False     OK
         Right Said Fred - I_m Too Sexy _Original Mix - 2006 Version_.wav                        Taylor Swift - Look What You Made Me Do.wav               Vanilla Ice - Ice Ice Baby _Official Music Video_.wav 0.9954  False     OK
          

,ori,expected,predicted,score,match,status
0,The Gap Band - Oops Upside Your Head.wav,Mark Ronson - Uptown Funk _Official Video_ ft_...,Bandido - Vamos Amigos _Eurodance Version_.wav,0.9864,False,OK
1,Jennifer Rush - The Power Of Love _Official Vi...,Céline Dion - The Power Of Love _Official Rema...,Céline Dion - The Power Of Love _Official Rem...,0.9968,False,OK
2,Without you - Badfinger.wav,Mariah Carey - Without You _Official Lyric Vid...,Mariah Carey - Without You _Official Lyric Vid...,0.9917,True,OK
3,_Official Audio_ 이정현_Lee Jung-hyun_ - 와.wav,Bandido - Vamos Amigos _Eurodance Version_.wav,None,NaN,False,NOT_FOUND
4,Queen - Under Pressure _Official Video_.wav,Vanilla Ice - Ice Ice Baby _Official Music Vid...,Smoke On The Water _2024 Remastered_.wav,0.9889,False,OK
5,Joyful Noise - Official Music Video_ FLAME fea...,Katy Perry - Dark Horse _Lyrics_ ft_ Juicy J.wav,Vanilla Ice - Ice Ice Baby _Official Music Vid...,0.9921,False,OK
6,Astrud Gilberto - Maria Quiet.wav,Smoke On The Water _2024 Remastered_.wav,Juice WRLD - Lucid Dreams _Official Music Vide...,0.9928,False,OK
7,Crazy Frog - Axel F _Official Video_.wav,_DANCE_ 싸이 _PSY_ - 챔피언.wav,Ed Sheeran - Bad Habits _Official Lyric Video_...,0.9776,False,OK
8,Sting - Shape of My Heart _Official Music Vide...,Juice WRLD - Lucid Dreams _Official Music Vide...,Juice WRLD - Lucid Dreams _Official Music Vide...,0.9949,True,OK
9,September - Cry For You.wav,Ed Sheeran - Bad Habits _Official Lyric Video_...,Juice WRLD - Lucid Dreams _Official Music Vide...,0.9928,False,OK


In [ ]:
print({"success":true,"filename":"audio.wav","analysis_timestamp":"2026-01-29T00:11:37.906731Z","processing72,"segments":[{"start":0.0,"end":15.36,"start_formatted":"00:00.000","end_formatted":"00:15.361","duel":"chorus"},{"start":15.36,"end":28.68,"start_formatted":"00:15.361","end_formatted":"00:28.681","dbel":"verse"},{"start":28.68,"end":33.48,"start_formatted":"00:28.681","end_formatted":"00:33.481","dl":"outro"}],"boundaries":[{"timestamp":0.0,"timestamp_formatted":"00:00.000","from_label":null,"to_ltimestamp":15.36,"timestamp_formatted":"00:15.361","from_label":"chorus","to_label":"verse"},{"timestamp_formatted":"00:28.681","from_label":"verse","to_label":"outro"},{"timestamp":33.48,"timestamp_for","from_label":"outro","to_label":"end"}],"statistics":{"total_segments":3,"total_duration":33.48,"unrus","verse","outro"],"label_counts":{"chorus":1,"verse":1,"outro":1},"label_durations":{"chorus":15.outro":4.8},"label_percentages":{"chorus":45.9,"verse":39.8,"outro":14.3},"average_segment_duration":gment":{"label":"outro","duration":4.8,"start":28.68},"longest_segment":{"label":"chorus","duration":},"msa_format":"0.00 chorus\n15.36 verse\n28.68 outro\n33.48 end","structure_string":"chorus-verse-ou{"name":"SongFormer","version":"1.0","mode":"real","github":"https://github.com/ASLP-lab/SongFormer"})

SyntaxError: ':' expected after dictionary key (ipython-input-1401673349.py, line 1)

In [ ]:
import requests

# Надіслати файл
with open("/content/Bandido - Vamos Amigos _Eurodance Version_.wav", "rb") as f:
    response = requests.post(
        "http://13.220.223.194:8000/analyze",
        files={"file": f}
    )

result = response.json()

# Вивести структуру
print(result["structure"])
# intro-verse-chorus-verse-chorus-bridge-chorus-outro

# Вивести сегменти
for seg in result["segments"]:
    print(f"{seg['time']}  {seg['label']}")
# 00:00.000 → 00:15.240  intro
# 00:15.240 → 00:45.500  verse
# ...

verse-chorus-chorus
00:00.000 → 00:07.200  verse
00:07.200 → 00:20.881  chorus
00:20.881 → 00:35.281  chorus
